![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# M3L2 E14 - LangChain vs Manual: comparación completa (Resolution)

## Qué es este notebook

Este notebook no enseña un concepto nuevo. Es un **recorrido completo** por todo lo que viste en M3L2, poniendo lado a lado:

- **Sin LangChain**: la forma manual, usando el SDK de `openai` directamente (el estilo de M3L1).
- **Con LangChain**: el mismo resultado usando los componentes que fuiste armando en E00-E12.

La idea es que puedas ver, en un solo lugar, **qué reemplaza a qué** y **por qué conviene**.

## Mapa de lo que vamos a comparar

| Bloque | Concepto | Sin LangChain | Con LangChain | Relacionado con |
|---|---|---|---|---|
| 1 | El modelo | `client.chat.completions.create()` | `ChatOpenAI` | E00 |
| 2 | El prompt | f-string manual | `ChatPromptTemplate` | E01 |
| 3 | La salida | `response.choices[0].message.content` | `StrOutputParser` | E02 |
| 4 | Composición | funciones encadenadas a mano | LCEL (`\|`) | E03 |
| 5 | Memoria | lista de mensajes manual | `RunnableWithMessageHistory` | E04 |
| 6 | Tools | JSON schema + dispatch manual | `@tool` + `bind_tools` | E05, E06 |
| 7 | Embeddings | `client.embeddings.create()` + loop | `OpenAIEmbeddings` | E07 |
| 8 | Vector store | búsqueda por fuerza bruta | `FAISS` | E08 |
| 9 | Retriever | función que envuelve la búsqueda | `as_retriever()` | E09 |
| 10 | Pipeline RAG completo | script "legacy" de punta a punta | chain LCEL de punta a punta | E10, E11, E12 |

## Este notebook necesita API key de OpenAI y `faiss-cpu`

Es una versión resuelta y ejecutable: no hay TODOs, corré las celdas en orden.

## BLOQUE 0 — Setup

In [ ]:
# !pip install langchain langchain-openai langchain-community faiss-cpu

import os, getpass, math, json

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")

print("API key cargada.")

In [ ]:
# Dataset comun para todo el notebook (el mismo que usamos en E08, E09, E10, E11)
DOCS_EMPRESA = [
    "La politica de vacaciones es de 15 dias por ano.",
    "Los empleados tienen seguro medico incluido desde el primer dia.",
    "El horario de trabajo es de 9 a 18 con 1 hora de almuerzo.",
    "El trabajo remoto esta habilitado 3 dias por semana previa aprobacion del manager.",
    "Los bonos anuales se calculan en base al desempeno y se pagan en diciembre.",
]

print(f"{len(DOCS_EMPRESA)} documentos de referencia listos.")

## BLOQUE 1 — El modelo: llamada directa vs `ChatOpenAI`

**Sin LangChain**: el SDK de `openai` te devuelve un objeto `ChatCompletion`. El modelo, la temperatura y el formato de mensajes quedan hardcodeados en cada lugar donde llamas a la API.

**Con LangChain**: `ChatOpenAI` encapsula esa llamada en un objeto configurable. El resto del pipeline no necesita saber que hay un modelo de OpenAI del otro lado.

In [ ]:
# --- SIN LANGCHAIN ---
from openai import OpenAI

client = OpenAI()

def llamar_modelo_manual(pregunta: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": pregunta}],
        temperature=0,
    )
    return response.choices[0].message.content

respuesta_manual = llamar_modelo_manual("Decime solo la palabra: hola")
print(f"Tipo de retorno: {type(client.chat.completions.create).__name__} -> texto extraido a mano")
print(f"Respuesta: {respuesta_manual}")

In [ ]:
# --- CON LANGCHAIN ---
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

respuesta_langchain = llm.invoke("Decime solo la palabra: hola")
print(f"Tipo de retorno: {type(respuesta_langchain).__name__}")
print(f"Respuesta: {respuesta_langchain.content}")
print()
print("El modelo es ahora un objeto (llm). Se puede pasar como parametro a cualquier otro componente.")

| Aspecto | Sin LangChain | Con LangChain |
|---|---|---|
| Modelo | hardcodeado en cada llamada | objeto `llm` reutilizable |
| Cambiar de proveedor | reescribir la funcion | `ChatOpenAI` -> `ChatAnthropic` |
| Se puede componer con otras piezas | no directamente | si, via `\|` |

**Relacionado con**: E00 - LLM Wrapper.

## BLOQUE 2 — El prompt: f-string vs `ChatPromptTemplate`

**Sin LangChain**: el prompt es un string armado a mano. Las variables estan implicitas (hay que leer el codigo para saber que espera).

**Con LangChain**: `ChatPromptTemplate` declara las variables explicitamente y separa el `system` del `human`.

In [ ]:
# --- SIN LANGCHAIN ---
def construir_prompt_manual(contexto: str, pregunta: str) -> str:
    system_msg = "Eres un asistente de RRHH. Responde solo con el contexto dado."
    return system_msg + "\n\nContexto:\n" + contexto + "\n\nPregunta:\n" + pregunta

contexto_demo = DOCS_EMPRESA[0]
prompt_manual = construir_prompt_manual(contexto_demo, "Cuantos dias de vacaciones tengo?")
print(prompt_manual)

In [ ]:
# --- CON LANGCHAIN ---
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente de RRHH. Responde solo con el contexto dado."),
    ("human", "Contexto:\n{context}\n\nPregunta:\n{question}"),
])

print(f"Variables declaradas: {rag_prompt.input_variables}")

mensajes = rag_prompt.format_messages(context=contexto_demo, question="Cuantos dias de vacaciones tengo?")
for m in mensajes:
    print(f"[{m.type.upper()}] {m.content}")

| Aspecto | Sin LangChain | Con LangChain |
|---|---|---|
| Variables | implicitas (hay que leer el codigo) | explicitas: `prompt.input_variables` |
| Inspeccion | imposible sin ejecutar | `.format_messages()` antes de llamar al modelo |
| Reutilizacion | copiar y pegar el string | pasar el objeto `rag_prompt` |

**Relacionado con**: E01 - PromptTemplate.

## BLOQUE 3 — La salida: extraccion manual vs `StrOutputParser`

**Sin LangChain**: cada vez que llamas al modelo tenes que acordarte de hacer `response.choices[0].message.content`.

**Con LangChain**: `llm.invoke()` devuelve un `AIMessage` (con metadata). `StrOutputParser` extrae `.content` automaticamente como ultimo paso.

In [ ]:
# --- SIN LANGCHAIN ---
response_cruda = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Decime solo: test"}],
    temperature=0,
)
texto_manual = response_cruda.choices[0].message.content
print(f"Extraccion manual: response.choices[0].message.content -> '{texto_manual}'")

In [ ]:
# --- CON LANGCHAIN ---
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage

parser = StrOutputParser()

resultado_crudo = llm.invoke("Decime solo: test")
print(f"Sin parser: tipo={type(resultado_crudo).__name__}")

resultado_parseado = parser.invoke(resultado_crudo)
print(f"Con parser: tipo={type(resultado_parseado).__name__} -> '{resultado_parseado}'")

# Lo mas comun: ponerlo directo en la chain
chain_con_parser = llm | parser
print(f"\n(llm | parser).invoke(...) -> '{chain_con_parser.invoke('Decime solo: test')}'")

| Aspecto | Sin LangChain | Con LangChain |
|---|---|---|
| Extraer texto | `response.choices[0].message.content` en cada lugar | `StrOutputParser` al final de la chain |
| Riesgo | olvidar el `.content` en algun lugar | siempre `str` |

**Relacionado con**: E02 - Output Parser.

## BLOQUE 4 — Composición: pasos imperativos vs LCEL

**Sin LangChain**: para armar prompt -> modelo -> texto, escribis una funcion que hace los tres pasos a mano.

**Con LangChain**: LCEL (`prompt | llm | parser`) declara el mismo flujo en una linea, y de paso te da streaming, batch y trazabilidad gratis.

In [ ]:
# --- SIN LANGCHAIN: 4 pasos manuales ---
def responder_manual(pregunta: str) -> str:
    # Paso 1: armar el prompt
    prompt_text = "Eres un asistente util. Responde de forma concisa.\n\n" + pregunta
    # Paso 2: armar los mensajes en formato OpenAI
    messages = [{"role": "user", "content": prompt_text}]
    # Paso 3: llamar a la API
    response = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0)
    # Paso 4: extraer el texto
    return response.choices[0].message.content

print(responder_manual("Cual es la capital de Francia?"))

In [ ]:
# --- CON LANGCHAIN: 1 linea declarativa ---
prompt_simple = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente util. Responde de forma concisa."),
    ("human", "{pregunta}"),
])

chain_simple = prompt_simple | llm | parser

print(chain_simple.invoke({"pregunta": "Cual es la capital de Francia?"}))
print()
print("Capacidades que vienen gratis con LCEL:")
print(" - chain_simple.stream(...)  -> tokens uno por uno")
print(" - chain_simple.batch([...]) -> multiples preguntas en paralelo")

| Aspecto | Sin LangChain | Con LangChain |
|---|---|---|
| Lineas para entender el flujo | 4 (hay que leer la funcion completa) | 1 (`prompt \| llm \| parser`) |
| Streaming / batch | hay que implementarlo a mano | gratis con LCEL |
| Reemplazar una pieza | reescribir la funcion | cambiar 1 variable |

**Relacionado con**: E03 - LCEL Chain.

## BLOQUE 5 — Memoria conversacional: lista manual vs `RunnableWithMessageHistory`

**Sin LangChain**: vos mismo guardas la lista de mensajes y la re-envias completa en cada llamada.

**Con LangChain**: `RunnableWithMessageHistory` guarda y re-inyecta el historial automaticamente, separado por `session_id`.

In [ ]:
# --- SIN LANGCHAIN: historial como lista manual ---
historial_manual = []  # lista de dicts {"role": ..., "content": ...}

def chat_manual(pregunta: str) -> str:
    historial_manual.append({"role": "user", "content": pregunta})
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": "Eres un asistente util."}] + historial_manual,
        temperature=0,
    )
    respuesta = response.choices[0].message.content
    historial_manual.append({"role": "assistant", "content": respuesta})
    return respuesta

print(chat_manual("Me llamo Ana."))
print(chat_manual("Como me llamo?"))
print(f"\nMensajes guardados a mano: {len(historial_manual)}")

In [ ]:
# --- CON LANGCHAIN ---
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

historiales = {}

def obtener_historial(session_id: str):
    if session_id not in historiales:
        historiales[session_id] = InMemoryChatMessageHistory()
    return historiales[session_id]

prompt_memoria = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente util."),
    MessagesPlaceholder("historial"),
    ("human", "{pregunta}"),
])
cadena_memoria = prompt_memoria | llm | parser

chat_con_memoria = RunnableWithMessageHistory(
    cadena_memoria,
    obtener_historial,
    input_messages_key="pregunta",
    history_messages_key="historial",
)

config = {"configurable": {"session_id": "demo"}}
print(chat_con_memoria.invoke({"pregunta": "Me llamo Ana."}, config=config))
print(chat_con_memoria.invoke({"pregunta": "Como me llamo?"}, config=config))
print(f"\nMensajes guardados automaticamente: {len(historiales['demo'].messages)}")

| Aspecto | Sin LangChain | Con LangChain |
|---|---|---|
| Guardar historial | `.append()` manual despues de cada turno | automatico via `RunnableWithMessageHistory` |
| Multiples conversaciones | armar tu propio diccionario de listas | `session_id` |
| Inyectar el historial en el prompt | concatenarlo vos | `MessagesPlaceholder` |

**Relacionado con**: E04 - Chat con memoria.

## BLOQUE 6 — Tools: JSON schema manual vs `@tool` + `bind_tools`

**Sin LangChain**: escribis el schema JSON de la tool a mano, lo mandas en el parametro `tools`, y despues parseas `tool_calls` para decidir que funcion ejecutar.

**Con LangChain**: `@tool` genera el schema a partir del type hint y el docstring. `bind_tools()` se lo pasa al modelo.

In [ ]:
# --- SIN LANGCHAIN: schema JSON manual ---
def calculadora_manual(expresion: str) -> str:
    return str(eval(expresion))

tool_schema_manual = {
    "type": "function",
    "function": {
        "name": "calculadora_manual",
        "description": "Evalua una expresion matematica simple enviada como texto.",
        "parameters": {
            "type": "object",
            "properties": {"expresion": {"type": "string"}},
            "required": ["expresion"],
        },
    },
}

response_tool = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Cuanto es 21 * 5?"}],
    tools=[tool_schema_manual],
)

tool_calls_manual = response_tool.choices[0].message.tool_calls
print(f"El modelo pidio ejecutar: {tool_calls_manual[0].function.name}")
args_manual = json.loads(tool_calls_manual[0].function.arguments)
print(f"Argumentos: {args_manual}")
print(f"Resultado ejecutando la funcion a mano: {calculadora_manual(**args_manual)}")

In [ ]:
# --- CON LANGCHAIN ---
from langchain_core.tools import tool

@tool
def calculadora(expresion: str) -> str:
    """Evalua una expresion matematica simple enviada como texto."""
    return str(eval(expresion))

print(f"Schema generado automaticamente a partir del docstring y el type hint:")
print(f"  name: {calculadora.name}")
print(f"  description: {calculadora.description}")

llm_con_tools = llm.bind_tools([calculadora])
respuesta_tool = llm_con_tools.invoke("Cuanto es 21 * 5?")
print(f"\nTool call detectado: {respuesta_tool.tool_calls[0]['name']}")
print(f"Argumentos: {respuesta_tool.tool_calls[0]['args']}")
print(f"Resultado invocando la tool: {calculadora.invoke(respuesta_tool.tool_calls[0]['args'])}")

| Aspecto | Sin LangChain | Con LangChain |
|---|---|---|
| Definir el schema | JSON a mano, sincronizado con la funcion | `@tool` lo genera del docstring + type hints |
| Ejecutar la tool elegida | parsear `tool_calls`, hacer `json.loads`, llamar la funcion | `tool.invoke(args)` |
| Agente completo (loop de N tools) | codigo pegamento (ver M3L1) | `AgentExecutor` |

**Relacionado con**: E05 - Tools, E06 - Manual a LangChain (agente completo con `AgentExecutor`).

## BLOQUE 7 — Embeddings: API cruda vs `OpenAIEmbeddings`

**Sin LangChain**: llamas al endpoint de embeddings del SDK y comparas vectores con tu propia funcion de similitud del coseno.

**Con LangChain**: `OpenAIEmbeddings` da los mismos vectores con una interfaz estandar (`embed_query`, `embed_documents`) que despues conecta directo con los vector stores.

In [ ]:
# --- SIN LANGCHAIN ---
def embed_manual(texto: str) -> list:
    resp = client.embeddings.create(model="text-embedding-ada-002", input=texto)
    return resp.data[0].embedding

def cosine_similarity(v1: list, v2: list) -> float:
    dot = sum(a * b for a, b in zip(v1, v2))
    n1 = math.sqrt(sum(a * a for a in v1))
    n2 = math.sqrt(sum(b * b for b in v2))
    return dot / (n1 * n2) if n1 and n2 else 0.0

v_manual_1 = embed_manual("vacaciones")
v_manual_2 = embed_manual("licencia y dias libres")
print(f"Dimensiones: {len(v_manual_1)}")
print(f"Similitud 'vacaciones' vs 'licencia y dias libres': {cosine_similarity(v_manual_1, v_manual_2):.4f}")

In [ ]:
# --- CON LANGCHAIN ---
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

v_lc_1 = embeddings.embed_query("vacaciones")
v_lc_2 = embeddings.embed_query("licencia y dias libres")
print(f"Dimensiones: {len(v_lc_1)}")
print(f"Similitud (misma cosine_similarity de arriba): {cosine_similarity(v_lc_1, v_lc_2):.4f}")

vectores_docs = embeddings.embed_documents(DOCS_EMPRESA)
print(f"\n embed_documents() embebio los {len(vectores_docs)} documentos en un solo llamado.")

| Aspecto | Sin LangChain | Con LangChain |
|---|---|---|
| Generar un vector | `client.embeddings.create(...)` + extraer `.data[0].embedding` | `embeddings.embed_query(texto)` |
| Comparar textos | tu propia `cosine_similarity` | igual (LangChain no reinventa la matematica) |
| Conectar con un vector store | armar el indice vos mismo | `FAISS.from_texts(docs, embeddings)` |

**Relacionado con**: E07 - Embeddings.

## BLOQUE 8 — Vector store: fuerza bruta vs `FAISS`

**Sin LangChain**: para buscar el documento mas parecido, embebes todos los documentos una vez y despues comparas el vector de la pregunta contra cada uno con un loop.

**Con LangChain**: `FAISS` hace lo mismo pero con un indice optimizado, y ademas guarda el texto original junto al vector.

In [ ]:
# --- SIN LANGCHAIN: busqueda por fuerza bruta ---
vectores_manual = [embed_manual(doc) for doc in DOCS_EMPRESA]

def buscar_manual(query: str, k: int = 2) -> list:
    v_query = embed_manual(query)
    similitudes = [
        (cosine_similarity(v_query, v_doc), doc)
        for v_doc, doc in zip(vectores_manual, DOCS_EMPRESA)
    ]
    similitudes.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in similitudes[:k]]

resultados_manual = buscar_manual("Cuantos dias de vacaciones tengo?", k=2)
print("Busqueda manual (O(n), recorre todos los documentos):")
for r in resultados_manual:
    print(f"  - {r}")

In [ ]:
# --- CON LANGCHAIN ---
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_texts(DOCS_EMPRESA, embeddings)

docs_faiss = vectorstore.similarity_search("Cuantos dias de vacaciones tengo?", k=2)
print("Busqueda con FAISS (indice optimizado):")
for doc in docs_faiss:
    print(f"  - {doc.page_content}")

print(f"\nMismo resultado, pero FAISS escala a millones de documentos. El loop manual, no.")

| Aspecto | Sin LangChain | Con LangChain |
|---|---|---|
| Buscar el mas parecido | loop + `cosine_similarity` sobre todos los vectores | `vectorstore.similarity_search(query, k)` |
| Escala | O(n): se degrada con miles de documentos | indice optimizado para vecinos cercanos |
| Persistencia | serializar los vectores vos mismo | `save_local()` / `load_local()` |

**Relacionado con**: E08 - FAISS.

## BLOQUE 9 — Retriever: funcion manual vs `as_retriever()`

**Sin LangChain**: `buscar_manual()` ya es una funcion reutilizable, pero esta atada a como implementaste la busqueda (FAISS, fuerza bruta, lo que sea).

**Con LangChain**: `as_retriever()` expone una interfaz estandar (`.invoke(query)`) que es igual sin importar que vector store haya atras.

In [ ]:
# --- SIN LANGCHAIN: la funcion ya es "un retriever", pero no es intercambiable ---
docs_manual = buscar_manual("trabajo remoto", k=1)
print(f"buscar_manual('trabajo remoto') -> {docs_manual}")
print("Si mañana cambio la implementacion interna (FAISS, Pinecone, otra cosa),")
print("tengo que reescribir buscar_manual() y todo lo que la llama.")

In [ ]:
# --- CON LANGCHAIN ---
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

docs_retriever = retriever.invoke("trabajo remoto")
print(f"retriever.invoke('trabajo remoto') -> {[d.page_content for d in docs_retriever]}")
print()
print("Si mañana cambio 'vectorstore = FAISS(...)' por 'vectorstore = Chroma(...)',")
print("el retriever sigue respondiendo a .invoke() igual. Ademas, se puede conectar")
print("directamente en una chain LCEL con el operador |.")

| Aspecto | Sin LangChain | Con LangChain |
|---|---|---|
| Interfaz | la que vos definiste (`buscar_manual`) | estandar: `.invoke(query)` en cualquier retriever |
| Cambiar de vector store | reescribir la funcion | cambiar 1 linea (`vectorstore = ...`) |
| Conectar a una chain LCEL | no directamente | `retriever \| format_docs \| ...` |

**Relacionado con**: E09 - Retriever.

## BLOQUE 10 — El pipeline RAG completo: script "legacy" vs chain LCEL

Este es el bloque final: juntamos **todo** lo de arriba en dos implementaciones completas del mismo chatbot de RRHH, y las comparamos con las mismas preguntas.

In [ ]:
# --- SIN LANGCHAIN: pipeline RAG armado a mano, de punta a punta ---

def rag_manual(pregunta: str) -> str:
    # 1. Retrieval (reusando buscar_manual, que ya escribimos arriba)
    docs_relevantes = buscar_manual(pregunta, k=2)

    # 2. Construir el contexto
    contexto = "\n".join(docs_relevantes)

    # 3. Construir el prompt a mano
    prompt_text = (
        "Eres un asistente de RRHH. Responde usando solo el contexto dado.\n\n"
        f"Contexto:\n{contexto}\n\nPregunta:\n{pregunta}"
    )

    # 4. Llamar al modelo
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_text}],
        temperature=0,
    )

    # 5. Extraer el texto
    return response.choices[0].message.content


print("=== RAG manual ===")
print(rag_manual("Cuantos dias de vacaciones tengo?"))

In [ ]:
# --- CON LANGCHAIN: la misma logica, compuesta con LCEL ---
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs) -> str:
    return "\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | parser
)

print("=== RAG con LangChain ===")
print(rag_chain.invoke("Cuantos dias de vacaciones tengo?"))

In [ ]:
# --- Comparacion lado a lado con varias preguntas ---
preguntas_demo = [
    "Cuantos dias de vacaciones tengo?",
    "Puedo trabajar desde casa?",
    "Cuando se pagan los bonos?",
]

for p in preguntas_demo:
    print(f"Pregunta: {p}")
    print(f"  Manual     : {rag_manual(p)}")
    print(f"  LangChain  : {rag_chain.invoke(p)}")
    print()

### El mismo pipeline, dos implementaciones

| Etapa | Manual | LangChain |
|---|---|---|
| Ingestion | `vectores_manual = [embed_manual(d) ...]` | `FAISS.from_texts(docs, embeddings)` (E07, E08) |
| Retrieval | `buscar_manual(pregunta, k)` | `retriever.invoke(pregunta)` (E09) |
| Prompt | f-string concatenado | `ChatPromptTemplate` (E01) |
| Modelo | `client.chat.completions.create(...)` | `llm` (E00) |
| Salida | `response.choices[0].message.content` | `StrOutputParser` (E02) |
| Composicion | funcion `rag_manual()` de 5 pasos | `{...} \| prompt \| llm \| parser` (E03) |
| Debugging | agregar prints en cada paso | `retriever.invoke(...)`, `rag_prompt.format_messages(...)` por separado |

**Relacionado con**: E10 - RAG mini, E11 - RAG con memoria, E12 - RAG desde cero.

## BLOQUE 11 — Checks automáticos

In [ ]:
def run_checks():
    # Bloque 1-3: modelo, prompt, parser
    assert isinstance(llm.invoke("test"), AIMessage)
    assert isinstance((llm | parser).invoke("test"), str)

    # Bloque 4: LCEL
    assert chain_simple.invoke({"pregunta": "di hola"})

    # Bloque 5: memoria
    assert "demo" in historiales
    assert len(historiales["demo"].messages) >= 4

    # Bloque 6: tools
    assert calculadora.name == "calculadora"

    # Bloque 7-9: embeddings, vector store, retriever
    assert len(embeddings.embed_query("test")) > 100
    assert len(vectorstore.similarity_search("vacaciones", k=2)) == 2
    assert len(retriever.invoke("vacaciones")) >= 1

    # Bloque 10: RAG completo, ambas versiones deben responder lo mismo en esencia
    r_manual = rag_manual("Cuantos dias de vacaciones tengo?")
    r_lc = rag_chain.invoke("Cuantos dias de vacaciones tengo?")
    assert "15" in r_manual
    assert "15" in r_lc

    print("M3L2 E14 checks passed")


run_checks()

## Cierre

Recorriste **todo M3L2** dos veces: una a mano y otra con LangChain.

### Lo que te queda de este notebook

1. LangChain **no inventa conceptos nuevos**: modelo, prompt, parser, memoria, tools, embeddings, vector store y retriever existen igual sin el framework.
2. Lo que LangChain agrega es una **interfaz estandar** (`.invoke()` en todos lados) y **composicion declarativa** (`\|`).
3. El costo de la version manual no se nota con 5 documentos y una sola conversacion. Se nota cuando el sistema crece: mas documentos, mas tools, mas memoria, mas modelos.
4. Si podes explicar cada bloque de este notebook **sin mirar el codigo de LangChain**, entendiste el framework de verdad — porque sabes que hay abajo de cada abstraccion.

### Si queres seguir practicando

- Reemplaza `FAISS` por `Chroma` en el bloque 8 y confirma que el resto del pipeline no cambia.
- Agrega una segunda tool en el bloque 6 y arma un `AgentExecutor` (como en E06).
- Combina memoria (bloque 5) con el RAG completo (bloque 10), como en E11.